In [ ]:
from time import perf_counter

from ssf2_rl.policy.bots import Agent, ZeroBot, FollowBot, ScriptedBot
from ssf2_rl.game.catalog import Character, Stage
from ssf2_rl.game.players import CPU, Human
from ssf2_rl import NOOP, LEFT, RIGHT, DOWN, SPECIAL, ATTACK
from ssf2_rl.env.gym_env import SSF2Env

# Restart the notebook kernel after rebuilding/reinstalling ssf2_rl so imports
# use the current bridge protocol implementation.

In [13]:
# --- Lockstep stepping: debugger-safe one-frame transitions ------------------
# reset() returns while the game is silently paused. The synchronous mode
# directly executes one simulation tick per Python action; it does not repeat
# actions or skip policy observations.

lockstep_env = SSF2Env(
    lockstep=True,
    lockstep_mode="synchronous",
    state_transport="json",  # default; benchmark both transports below
)
obs, info = lockstep_env.reset(
    players={
        1: ZeroBot(Character.Marth),
        2: ZeroBot(Character.ZeroSuitSamus),
    },
    stage=Stage.bf,
)
assert info["lockstep"] and info["paused"]
frame_before = info["frame"]
print(f"Paused at frame {frame_before}. Set a breakpoint here if desired.")

obs, reward, terminated, truncated, info = lockstep_env.step(0)
assert info["paused"]
assert info["frame"] == frame_before + 1
print(f"One exact step completed at frame {info['frame']}.")

# Keep lockstep_env open for manual debugger stepping. Close it before running
# the benchmark cell, because the game intentionally supports one client.

[ssf2_rl] Launching SSF2 via /Users/cachemiss/Developer/AIRSDK_51.3.3/bin/adl (log: /Users/cachemiss/Documents/projects/reflash2-fork/reflash2/.macos/adl.log) ...
[ssf2_rl] Game is up; bridge listening on 127.0.0.1:4567.
Paused at frame 1. Set a breakpoint here if desired.
One exact step completed at frame 2.


In [42]:
lockstep_env.step()

(array([-3.29625e-01,  1.09375e-01,  0.00000e+00,  0.00000e+00,
         1.00000e+00,  0.00000e+00,  1.00000e+00,  1.00000e+00,
         0.00000e+00,  1.00000e+00, -1.00000e+00, -1.00000e+00,
        -1.00000e+00,  0.00000e+00, -1.00000e+00, -1.00000e+00,
         3.34625e-01,  1.09500e-01,  0.00000e+00,  0.00000e+00,
        -1.00000e+00,  0.00000e+00,  1.00000e+00,  1.00000e+00,
         0.00000e+00,  1.00000e+00, -1.00000e+00, -1.00000e+00,
        -1.00000e+00,  0.00000e+00, -1.00000e+00, -1.00000e+00,
         6.64250e-01,  1.25000e-04,  6.64250e-01,  1.00000e+00,
         3.10000e-04,  1.00000e+00], dtype=float32),
 0.0,
 False,
 False,
 {'frame': 31,
  'paused': True,
  'lockstep': True,
  'lockstep_mode': 'synchronous',
  'state_transport': 'json',
  'step_seconds': 0.010332749981898814,
  'simulation_fps': 96.7796570856574,
  'me': {'ground': 1,
   'atkExec': 0,
   'shielding': 0,
   'facing': 1,
   'hanging': 0,
   'dead': 0,
   'nxs': 0,
   'y': 43.75,
   'damage': 0,
   'x'

In [43]:
# --- JSON-minimal versus binary-v3 benchmark ---------------------------------
# One policy action still produces exactly one resulting observation.
try:
    lockstep_env.close()
except NameError:
    pass

N = 1000
results = {}
for transport in ("json", "binary-v3"):
    bench_env = SSF2Env(
        lockstep=True,
        lockstep_mode="synchronous",
        state_transport=transport,
        step_timeout=5.0,
    )
    try:
        obs, info = bench_env.reset(
            players={
                1: ZeroBot(Character.Marth),
                2: ZeroBot(Character.ZeroSuitSamus),
            },
            stage=Stage.bf,
        )
        previous = info["frame"]
        for _ in range(25):
            obs, reward, terminated, truncated, info = bench_env.step(0)
            assert info["paused"] and info["frame"] == previous + 1
            previous = info["frame"]

        started = perf_counter()
        for _ in range(N):
            obs, reward, terminated, truncated, info = bench_env.step(0)
            assert info["paused"] and info["frame"] == previous + 1
            previous = info["frame"]
        elapsed = perf_counter() - started
        results[transport] = N / elapsed
    finally:
        bench_env.close()

for transport, fps in results.items():
    print(f"{transport:9s}: {fps:6.1f} simulation FPS ({fps / 30:4.2f}x real time)")

json     :   40.0 simulation FPS (1.33x real time)
binary-v3:   31.6 simulation FPS (1.05x real time)
